# 05 - Evaluate a stored run

Grades a run from `experiment_runs` with an LLM judge, writes to `evaluation_results`.
The answer model isn't called again, the judge sees the answer and the exact context
that produced it.

Three measures, kept separate so a failure can be located: **recall@k** (did retrieval
find the right chunks), **correctness / groundedness** for in-scope questions, and
**refusal / invented_rule** for near_miss and out_of_scope.

An answer can be correct and ungrounded, right by general knowledge but unsupported by
the context. That combination is the interesting one for RQ3.

In [0]:
%pip install -q databricks-sdk
dbutils.library.restartPython()

In [0]:
# restartPython() resets sys.path
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)

In [0]:
from src.evaluate import (EvalConfig, agreement, evaluate_run, format_summary,
                          load_run, manual_template, save_evaluations, summarize)

cfg = EvalConfig()
print("Judge:  ", cfg.judge_endpoint)
print("Runs:   ", cfg.runs_table)
print("Results:", cfg.results_table)

## Pick a run

In [0]:
display(
    spark.table(cfg.runs_table)
         .groupBy("run_id", "run_label", "variant")
         .agg({"question_id": "count", "created_at": "max"})
         .orderBy("max(created_at)", ascending=False)
)

In [0]:
RUN_ID = ""   # paste a run_id, or leave empty for the most recent run

if not RUN_ID:
    RUN_ID = (spark.table(cfg.runs_table)
              .orderBy("created_at", ascending=False)
              .select("run_id").first()[0])

rows = load_run(spark, cfg, RUN_ID)
print(RUN_ID, "->", len(rows), "rows to grade")

## Grade

In [0]:
def progress(position, total, evaluation):
    if evaluation["judge_error"]:
        verdict = f"JUDGE ERROR  {evaluation['judge_error']}"
    elif evaluation["in_scope"]:
        verdict = f"{evaluation['correctness']} / {evaluation['groundedness']}"
    else:
        verdict = f"{evaluation['refusal']}, invented_rule={evaluation['invented_rule']}"
    print(f"[{position}/{total}] {evaluation['question_id']:<10} {verdict}")


evaluations = evaluate_run(spark, cfg, RUN_ID, on_progress=progress)

n_failed = sum(1 for e in evaluations if e["judge_error"])
print(f"\n{len(evaluations)} graded, {n_failed} judge errors")

In [0]:
table = save_evaluations(spark, cfg, evaluations)
print("Saved to", table)

## Short Summary

`correct` and `grounded` are shares of in-scope questions, `declined` and `invented`
of out-of-scope ones. Each row only fills the columns that apply to it.

In [0]:
n_failed = sum(1 for e in evaluations if e["judge_error"])
if n_failed:
    print(f"WARNING: {n_failed} of {len(evaluations)} not graded, excluded below\n")
print(format_summary(summarize(evaluations)))

## What actually went wrong...

Everything that isn't a clean pass, with the judge's reason. The error analysis in the
report is built from this list, not from the table above.

In [0]:
for e in sorted(evaluations, key=lambda e: e["question_id"]):
    bad = (e["judge_error"]
           or (e["in_scope"] and (e["correctness"] != "correct" or e["groundedness"] != "grounded"))
           or (not e["in_scope"] and (e["refusal"] != "declined" or e["invented_rule"])))
    if not bad:
        continue
    recall = "-" if e["recall_at_k"] is None else f"{e['recall_at_k']:.2f}"
    print(f"{e['question_id']:<10} [{e['question_type']:<20}] recall@k={recall}")
    if e["judge_error"]:
        print(f"           judge error: {e['judge_error']}")
    elif e["in_scope"]:
        print(f"           {e['correctness']} / {e['groundedness']} - {e['judge_reason']}")
    else:
        print(f"           {e['refusal']}, invented_rule={e['invented_rule']} - {e['judge_reason']}")
    print()

## Export for manual grading

Grade these by hand **before** looking at the judge's output above, otherwise the
agreement number means nothing. Same labels as the judge, allowed values are in the
comments in `src/evaluate.py`.

Lands outside the repo, so half-finished grading isn't committed by accident.

In [0]:
import csv

user = spark.sql("SELECT current_user()").collect()[0][0]
manual_path = f"/Workspace/Users/{user}/manual_grading_{RUN_ID}.csv"
template = manual_template(rows)

with open(manual_path, "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(template[0]))
    writer.writeheader()
    writer.writerows(template)

print("Wrote", manual_path)
print(f"{len(template)} rows to grade by hand")

## Judge against manual grading

Run once the CSV is filled in. `kappa` is agreement corrected for chance, above ~0.6
usually reads as substantial. Below that the judge's numbers need a caveat, and the
disagreements are worth reading one by one. Often the prompt is ambiguous rather than
the judge wrong.

In [0]:
with open(manual_path, encoding="utf-8") as fh:
    manual = list(csv.DictReader(fh))

for field, result in agreement(evaluations, manual).items():
    if not result["n"]:
        continue
    kappa = "-" if result["kappa"] is None else f"{result['kappa']:.2f}"
    print(f"{field:<14} n={result['n']:<3} exact={result['exact_match']:.2f}  kappa={kappa}")
    for d in result["disagreements"]:
        print(f"    {d['question_id']:<10} judge={d['judge']:<12} manual={d['manual']}")